In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [10]:
df = pd.read_csv('Statistics for decision making/property.csv')
print(df.head())

       Suburb           Address  Rooms Type      Price Method SellerG  \
0  Abbotsford      85 Turner St      2    h  1480000.0      S  Biggin   
1  Abbotsford   25 Bloomburg St      2    h  1035000.0      S  Biggin   
2  Abbotsford      5 Charles St      3    h  1465000.0     SP  Biggin   
3  Abbotsford  40 Federation La      3    h   850000.0     PI  Biggin   
4  Abbotsford       55a Park St      4    h  1600000.0     VB  Nelson   

        Date  Distance  Postcode  ...  Bathroom  Car  Landsize  BuildingArea  \
0  3/12/2016       2.5    3067.0  ...       1.0  1.0     202.0           NaN   
1  4/02/2016       2.5    3067.0  ...       1.0  0.0     156.0          79.0   
2  4/03/2017       2.5    3067.0  ...       2.0  0.0     134.0         150.0   
3  4/03/2017       2.5    3067.0  ...       2.0  1.0      94.0           NaN   
4  4/06/2016       2.5    3067.0  ...       1.0  2.0     120.0         142.0   

   YearBuilt  CouncilArea Lattitude  Longtitude             Regionname  \
0     

In [13]:
# Filter for Altona suburb
altona_prices = df[df['Suburb'].str.lower() == 'altona']['Price'].dropna()

In [14]:
# Perform 1-sample t-test
t_stat, p_val_two_tailed = stats.ttest_1samp(altona_prices, 800000)

In [18]:
# Since we are testing if it has *increased*, it's a one-tailed test.
# We divide the two-tailed p-value by 2 (and check if t_stat > 0).
p_val_one_tailed = p_val_two_tailed / 2 if t_stat > 0 else 1 - (p_val_two_tailed / 2)

print # (Questions 1 Results)
print(f"Altona Sample Size: {len(altona_prices)}")
print(f"Sample Mean Price: ${altona_prices.mean():,.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"One-tailed p-value: {p_val_one_tailed:.4f}")

# Decision
alpha = 0.05
if p_val_one_tailed < alpha:
    print("Conclusion: Reject the Null Hypothesis. The typical property price has significantly increased.")
else:
    print("Conclusion: Fail to reject the Null Hypothesis. No significant evidence that the price has increased.")

Altona Sample Size: 74
Sample Mean Price: $834,830.41
t-statistic: 1.0277
One-tailed p-value: 0.1537
Conclusion: Fail to reject the Null Hypothesis. No significant evidence that the price has increased.


In [21]:
# 1. Convert Date column to datetime (adjust format if needed, e.g., dayfirst=True)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# 2. Filter for the year 2016
df_2016 = df[df['Date'].dt.year == 2016].copy()

# 3. Define seasons based on assignment rules
# Winter: Oct (10), Nov (11), Dec (12), Jan (1), Feb (2), Mar (3)
df_2016['season'] = df_2016['Date'].dt.month.apply(lambda m: 'Winter' if m in [10, 11, 12, 1, 2, 3] else 'Summer')

# 4. Separate the groups
summer_prices = df_2016[df_2016['season'] == 'Summer']['Price'].dropna()
winter_prices = df_2016[df_2016['season'] == 'Winter']['Price'].dropna()

# 5. Independent 2-Sample T-test (Two-tailed, as it asks if there is "any difference")
t_stat_2, p_val_2 = stats.ttest_ind(summer_prices, winter_prices, equal_var=False) # Welch's t-test

print("\n--- Question 2 Results ---")
print(f"Summer Mean Price: ${summer_prices.mean():,.2f} (n={len(summer_prices)})")
print(f"Winter Mean Price: ${winter_prices.mean():,.2f} (n={len(winter_prices)})")
print(f"p-value: {p_val_2:.4f}")

if p_val_2 < alpha:
    print("Conclusion: Reject $H_0$. There is a significant difference between summer and winter prices.")
else:
    print("Conclusion: Fail to reject $H_0$. No significant difference detected.")


--- Question 2 Results ---
Summer Mean Price: $1,048,054.73 (n=4036)
Winter Mean Price: $1,116,647.59 (n=2300)
p-value: 0.0001
Conclusion: Reject $H_0$. There is a significant difference between summer and winter prices.


In [27]:
# Filter data for Abbotsford
abbotsford_df = df[df['Suburb'].str.lower() == 'abbotsford'].copy()

print(f"Total properties in Abbotsford: {len(abbotsford_df)}")

# --- Question 3 ---
# Probability of NO car parking space (car == 0)
total_abbotsford = len(abbotsford_df['Car'].dropna())
no_Car_count = len(abbotsford_df[abbotsford_df['Car'] == 0])
p_no_Car = no_Car_count / total_abbotsford

# Binomial calculation: 3 out of 10 properties have no car space
from scipy.stats import binom
prob_q3 = binom.pmf(k=3, n=10, p=p_no_Car)
print(f"Q3 Answer: {prob_q3:.3f}")

# --- Question 4 ---
# Probability of a property having 3 rooms
rooms_3_count = len(abbotsford_df[abbotsford_df['Rooms'] == 3])
prob_q4 = rooms_3_count / len(abbotsford_df['Rooms'].dropna())
print(f"Q4 Answer: {prob_q4:.3f}")

# --- Question 5 ---
# Probability of a property having 2 bathrooms
bathroom_2_count = len(abbotsford_df[abbotsford_df['Bathroom'] == 2])
prob_q5 = bathroom_2_count / len(abbotsford_df['Bathroom'].dropna())
print(f"Q5 Answer: {prob_q5:.3f}")

Total properties in Abbotsford: 56
Q3 Answer: 0.260
Q4 Answer: 0.357
Q5 Answer: 0.339


In [30]:
richmond_Prices = df[df['Suburb'].str.lower() == 'richmond']['Price'].dropna()

t_stat_rich, p_val_rich = stats.ttest_1samp(richmond_prices, 1000000)

print("--- Question 6 Results ---")
print(f"Richmond Sample Mean: ${richmond_Prices.mean():,.2f}")
print(f"Test Statistic (t): {t_stat_rich:.4f}")
print(f"p-value: {p_val_rich:.4f}")

if p_val_rich < 0.05:
    print("Conclusion: Reject H0. The average property price in Richmond is significantly different from $1,000,000.")
else:
    print("Conclusion: Fail to reject H0. No significant difference from the claim.")

--- Question 6 Results ---
Richmond Sample Mean: $1,083,564.42
Test Statistic (t): 2.5795
p-value: 0.0104
Conclusion: Reject H0. The average property price in Richmond is significantly different from $1,000,000.


with_car = df[df['Car'] > 0]['Price'].dropna()
without_car = df[df['Car'] == 0]['Price'].dropna()

t_stat_car, p_val_car = stats.ttest_ind(with_car, without_car, equal_var=False, alternative='greater')

print("--- Question 7 Results ---")
print(f"Mean with Car Parking: ${with_car.mean():,.2f}")
print(f"Mean without Car Parking: ${without_car.mean():,.2f}")
print(f"p-value: {p_val_car:.4f}")

In [ ]:
#Business Implications for Developers: If the p-value <0.05, properties with parking spaces command a statistically higher premium. Developers should prioritize building parking spaces to maximize project ROI.

In [36]:
# Ensure categorical variables are clean and drop missing values for these columns
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
# If it's explicitly a ValueWarning from statsmodels:
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.filterwarnings('ignore', category=ValueWarning)

# Now run your ANOVA code safely
model = ols('Price ~ C(Suburb) + C(Type) + C(Suburb):C(Type)', data=anova_df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                         sum_sq       df          F         PR(>F)
C(Suburb)          3.659365e+15    313.0  72.849995   0.000000e+00
C(Type)            2.040596e+12      2.0   6.357630   1.169971e-02
C(Suburb):C(Type)  6.470034e+14    626.0   6.440216  2.505093e-263
Residual           2.068639e+15  12890.0        NaN            NaN


Question 9: p-Value Interpretation

What it indicates: A p-value of 0.032 means there is a 3.2% probability of observing this big of a price difference between the two suburbs purely by random chance under the assumption that the true prices are identical.
Should H0 be rejected? Yes, because 0.032<α=0.05.
Business Stakeholder interpretation: There is a statistically significant difference in property prices between these two suburbs. Marketing, valuation, and investing decisions should treat these locations as distinctly different tiers.

In [37]:
premium_bath = df[df['Bathroom'] > 2]['Price'].dropna()
standard_bath = df[df['Bathroom'] <= 2]['Price'].dropna()

t_stat_bath, p_val_bath = stats.ttest_ind(premium_bath, standard_bath, equal_var=False, alternative='greater')

print("--- Question 10 Results ---")
print(f"Mean Price (>2 Bathrooms): ${premium_bath.mean():,.2f}")
print(f"Mean Price (<=2 Bathrooms): ${standard_bath.mean():,.2f}")
print(f"p-value: {p_val_bath:.4f}")

--- Question 10 Results ---
Mean Price (>2 Bathrooms): $1,882,824.20
Mean Price (<=2 Bathrooms): $1,007,347.94
p-value: 0.0000


Recommendation to Policymakers: If the p-value <0.05, the claim is verified. Policymakers should factor this in when designing affordable housing codes or layout guidelines, understanding that limiting bathroom count can be a viable lever to keep structural market costs down for low-and-middle-income buyers.